# Fase 3 — Voting Ensemble: LR + RF + XGBoost

Este notebook combina los mejores modelos obtenidos previamente para cada variante dialectal:

- **Logistic Regression (LR)**
- **Random Forest (RF)**
- **XGBoost (XGB)**

Cada modelo conserva **su propio preprocessing y sus hiperparámetros finales**.

## Metodología

1. Las configuraciones de LR, RF y XGBoost quedan **fijadas** a partir de los experimentos anteriores.
2. Se usa **Nested CV 5×5** sobre el train para escoger únicamente la estrategia de combinación:
   - Hard voting.
   - Soft voting con pesos iguales.
   - Soft voting ponderado.
3. Se reportan los **5 resultados Outer**, la media y la desviación estándar.
4. Se selecciona la configuración final de Voting usando todo el train con 5-fold CV.
5. Recién entonces se carga el **test oficial** y se evalúa una única configuración final por variante.

> La estimación de este notebook evalúa el ensemble condicionado a las configuraciones base previamente seleccionadas. No vuelve a optimizar LR, RF ni XGBoost. El test oficial sigue completamente separado de la selección del Voting.

## Implementación del Voting

Como en España LR, RF y XGBoost utilizan preprocessings distintos, cada pipeline recibe su propia versión del texto y las predicciones se combinan manualmente. La regla de hard/soft voting es la misma que usaría un `VotingClassifier`, pero sin obligar a los tres modelos a compartir la misma entrada textual.


## 1. Imports y configuración general

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_DIR = "../data"
RESULTS_DIR = f"{DATA_DIR}/voting_lr_rf_xgb"
MODELS_DIR = f"{RESULTS_DIR}/modelos"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

VARIANTES = ["mx", "es", "cu"]

FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun",
]

PREPROCESAMIENTOS = {
    "normal": "",
    "stem": "_stem",
    "lemma": "_lemma",
}

STOP_WORDS = stopwords.words("spanish")

N_OUTER = 5
N_INNER = 5

print(f"Nested CV del Voting: {N_OUTER}×{N_INNER}")


Nested CV del Voting: 5×5


## 2. Configuraciones base fijadas

Estas configuraciones corresponden a los mejores modelos finales obtenidos en los notebooks anteriores.

- LR y RF conservan TF-IDF + características lingüísticas.
- XGBoost conserva su TF-IDF, preprocessing e hiperparámetros finales.
- En este notebook no se vuelven a optimizar los modelos base.


In [2]:
CONFIG_MODELOS = {
    "mx": {
        "LR": {
            "prep": "normal",
            "ngram_range": (1, 2),
            "C": 1.0,
            "solver": "liblinear",
        },
        "RF": {
            "prep": "normal",
            "ngram_range": (1, 2),
            "n_estimators": 200,
            "max_depth": 20,
            "min_samples_split": 5,
        },
        "XGB": {
            "prep": "normal",
            "params": {
                "n_estimators": 650,
                "max_depth": 3,
                "learning_rate": 0.02992226863094147,
                "min_child_weight": 3,
                "subsample": 0.7015929604579353,
                "colsample_bytree": 0.6700163019309586,
                "gamma": 3.206142668165488,
                "reg_alpha": 0.6992470454250899,
                "reg_lambda": 0.15834451867267532,
                "scale_pos_weight": 1.7909291838347168,
            },
        },
    },
    "es": {
        "LR": {
            "prep": "stem",
            "ngram_range": (1, 2),
            "C": 1.0,
            "solver": "liblinear",
        },
        "RF": {
            "prep": "lemma",
            "ngram_range": (1, 1),
            "n_estimators": 300,
            "max_depth": None,
            "min_samples_split": 5,
        },
        "XGB": {
            "prep": "normal",
            "params": {
                "n_estimators": 600,
                "max_depth": 3,
                "learning_rate": 0.013567662411096632,
                "min_child_weight": 3,
                "subsample": 0.7526965914574834,
                "colsample_bytree": 0.5509446456627474,
                "gamma": 4.67042234219341,
                "reg_alpha": 1.039096427652307,
                "reg_lambda": 0.2471044820643491,
                "scale_pos_weight": 1.4410971672069746,
            },
        },
    },
    "cu": {
        "LR": {
            "prep": "stem",
            "ngram_range": (1, 1),
            "C": 1.0,
            "solver": "liblinear",
        },
        "RF": {
            "prep": "stem",
            "ngram_range": (1, 1),
            "n_estimators": 200,
            "max_depth": 10,
            "min_samples_split": 5,
        },
        "XGB": {
            "prep": "stem",
            "params": {
                "n_estimators": 650,
                "max_depth": 4,
                "learning_rate": 0.021725423114650193,
                "min_child_weight": 3,
                "subsample": 0.7751612513093522,
                "colsample_bytree": 0.5344434552671052,
                "gamma": 7.277132342194694,
                "reg_alpha": 0.045093940109743186,
                "reg_lambda": 0.5701154663714562,
                "scale_pos_weight": 1.668412382733185,
            },
        },
    },
}

filas_cfg = []
for variante, cfgs in CONFIG_MODELOS.items():
    for modelo, cfg in cfgs.items():
        filas_cfg.append({
            "variante": variante,
            "modelo": modelo,
            "preprocessing": cfg["prep"],
        })

display(pd.DataFrame(filas_cfg))


,variante,modelo,preprocessing
0,mx,LR,normal
1,mx,RF,normal
2,mx,XGB,normal
3,es,LR,stem
4,es,RF,lemma
5,es,XGB,normal
6,cu,LR,stem
7,cu,RF,stem
8,cu,XGB,stem


## 3. Carga y validación de normal / stem / lemma

In [3]:
def cargar_split(variante, prep, split="train"):
    sufijo = PREPROCESAMIENTOS[prep]
    ruta = f"{DATA_DIR}/{split}_clean{sufijo}_{variante}.csv"
    return pd.read_csv(ruta)


def preparar_xy(df):
    columnas = ["MESSAGE_CLEAN"] + FEATURE_COLS
    faltantes = [c for c in columnas + ["IS_IRONIC"] if c not in df.columns]

    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")

    X = df[columnas].copy()
    y = df["IS_IRONIC"].astype(int).reset_index(drop=True)

    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[FEATURE_COLS] = X[FEATURE_COLS].fillna(0)
    X = X.reset_index(drop=True)

    return X, y


def cargar_todos_preps(variante, split="train"):
    X_por_prep = {}
    y_ref = None
    n_ref = None

    for prep in PREPROCESAMIENTOS:
        df = cargar_split(variante, prep, split=split)
        X, y = preparar_xy(df)

        if y_ref is None:
            y_ref = y.copy()
            n_ref = len(y)
        else:
            if len(y) != n_ref:
                raise ValueError(
                    f"{variante}/{split}/{prep}: {len(y)} filas; se esperaban {n_ref}."
                )

            if not np.array_equal(y.to_numpy(), y_ref.to_numpy()):
                raise ValueError(
                    f"{variante}/{split}: etiquetas no alineadas entre normal/stem/lemma."
                )

        X_por_prep[prep] = X

    return X_por_prep, y_ref


# Aquí se valida únicamente TRAIN.
for variante in VARIANTES:
    _, y = cargar_todos_preps(variante, split="train")
    print(
        variante.upper(),
        "| n =", len(y),
        "| clase 0 =", int((y == 0).sum()),
        "| clase 1 =", int((y == 1).sum()),
    )


MX | n = 2399 | clase 0 = 1599 | clase 1 = 800
ES | n = 2398 | clase 0 = 1598 | clase 1 = 800
CU | n = 2400 | clase 0 = 1600 | clase 1 = 800


## 4. Construcción de LR, RF y XGBoost

In [4]:
def crear_preprocesador_lr_rf(ngram_range):
    return ColumnTransformer([
        (
            "tfidf_word",
            TfidfVectorizer(
                stop_words=STOP_WORDS,
                max_features=10000,
                ngram_range=ngram_range,
            ),
            "MESSAGE_CLEAN",
        ),
        ("ling", "passthrough", FEATURE_COLS),
    ])


def crear_lr(variante):
    cfg = CONFIG_MODELOS[variante]["LR"]

    return Pipeline([
        ("prep", crear_preprocesador_lr_rf(cfg["ngram_range"])),
        (
            "clf",
            LogisticRegression(
                C=cfg["C"],
                solver=cfg["solver"],
                max_iter=10000,
                random_state=RANDOM_STATE,
                class_weight="balanced",
            ),
        ),
    ])


def crear_rf(variante):
    cfg = CONFIG_MODELOS[variante]["RF"]

    return Pipeline([
        ("prep", crear_preprocesador_lr_rf(cfg["ngram_range"])),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=cfg["n_estimators"],
                max_depth=cfg["max_depth"],
                min_samples_split=cfg["min_samples_split"],
                random_state=RANDOM_STATE,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ])


def crear_preprocesador_xgb():
    tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=20000,
        lowercase=False,
        sublinear_tf=True,
        dtype=np.float32,
    )

    return ColumnTransformer(
        transformers=[
            ("tfidf", tfidf, "MESSAGE_CLEAN"),
            ("linguisticas", "passthrough", FEATURE_COLS),
        ],
        remainder="drop",
    )


def crear_xgb(variante):
    params = CONFIG_MODELOS[variante]["XGB"]["params"]

    clf = XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )

    return Pipeline([
        ("features", crear_preprocesador_xgb()),
        ("xgb", clf),
    ])


def crear_modelos_base(variante):
    return {
        "LR": crear_lr(variante),
        "RF": crear_rf(variante),
        "XGB": crear_xgb(variante),
    }


def prep_modelo(variante, modelo):
    return CONFIG_MODELOS[variante][modelo]["prep"]


## 5. Candidatos de Voting

Se usa un espacio pequeño para reducir el riesgo de sobreajustar el ensemble.

### Hard voting
- Mayoría simple de LR, RF y XGBoost.

### Soft voting
Promedio ponderado de la probabilidad de la clase irónica:

- `[1, 1, 1]`
- `[2, 1, 1]`
- `[1, 2, 1]`
- `[1, 1, 2]`
- `[2, 2, 1]`
- `[2, 1, 2]`
- `[1, 2, 2]`


In [5]:
CANDIDATOS_VOTING = [
    {"nombre": "hard_equal",   "tipo": "hard", "weights": (1, 1, 1)},
    {"nombre": "soft_equal",   "tipo": "soft", "weights": (1, 1, 1)},
    {"nombre": "soft_lr2",     "tipo": "soft", "weights": (2, 1, 1)},
    {"nombre": "soft_rf2",     "tipo": "soft", "weights": (1, 2, 1)},
    {"nombre": "soft_xgb2",    "tipo": "soft", "weights": (1, 1, 2)},
    {"nombre": "soft_lr_rf",   "tipo": "soft", "weights": (2, 2, 1)},
    {"nombre": "soft_lr_xgb",  "tipo": "soft", "weights": (2, 1, 2)},
    {"nombre": "soft_rf_xgb",  "tipo": "soft", "weights": (1, 2, 2)},
]

display(pd.DataFrame(CANDIDATOS_VOTING))


,nombre,tipo,weights
0,hard_equal,hard,"(1, 1, 1)"
1,soft_equal,soft,"(1, 1, 1)"
2,soft_lr2,soft,"(2, 1, 1)"
3,soft_rf2,soft,"(1, 2, 1)"
4,soft_xgb2,soft,"(1, 1, 2)"
5,soft_lr_rf,soft,"(2, 2, 1)"
6,soft_lr_xgb,soft,"(2, 1, 2)"
7,soft_rf_xgb,soft,"(1, 2, 2)"


## 6. Funciones de predicción y combinación

In [6]:
ORDEN_MODELOS = ["LR", "RF", "XGB"]


def probabilidad_clase_1(modelo, X):
    probs = modelo.predict_proba(X)
    clases = np.asarray(modelo.classes_)

    idx = np.where(clases == 1)[0]
    if len(idx) != 1:
        raise ValueError(
            f"No se encontró una única clase positiva 1. classes_={clases}"
        )

    return probs[:, idx[0]]


def aplicar_voting(candidato, preds_clase, probs_positivas):
    weights = np.asarray(candidato["weights"], dtype=float)

    if candidato["tipo"] == "hard":
        matriz = np.column_stack([
            preds_clase[m]
            for m in ORDEN_MODELOS
        ])
        return (matriz.sum(axis=1) >= 2).astype(int)

    if candidato["tipo"] == "soft":
        matriz_prob = np.column_stack([
            probs_positivas[m]
            for m in ORDEN_MODELOS
        ])

        prob_final = np.average(
            matriz_prob,
            axis=1,
            weights=weights,
        )

        return (prob_final >= 0.5).astype(int)

    raise ValueError(
        f"Tipo de voting desconocido: {candidato['tipo']}"
    )


def entrenar_y_predecir_base(
    variante,
    X_por_prep,
    y,
    idx_train,
    idx_val,
):
    modelos = crear_modelos_base(variante)

    preds = {}
    probs = {}

    for nombre in ORDEN_MODELOS:
        prep = prep_modelo(variante, nombre)

        X_train = (
            X_por_prep[prep]
            .iloc[idx_train]
            .reset_index(drop=True)
        )
        X_val = (
            X_por_prep[prep]
            .iloc[idx_val]
            .reset_index(drop=True)
        )

        y_train = y.iloc[idx_train].to_numpy()

        modelo = modelos[nombre]
        modelo.fit(X_train, y_train)

        preds[nombre] = modelo.predict(X_val)
        probs[nombre] = probabilidad_clase_1(modelo, X_val)

    return preds, probs


def seleccionar_mejor_voting_oof(
    variante,
    X_por_prep,
    y,
    indices_base,
    n_splits=N_INNER,
    seed=RANDOM_STATE,
):
    y_base = y.iloc[indices_base].reset_index(drop=True)

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    oof_pred = {
        m: np.zeros(len(indices_base), dtype=int)
        for m in ORDEN_MODELOS
    }
    oof_prob = {
        m: np.zeros(len(indices_base), dtype=float)
        for m in ORDEN_MODELOS
    }

    for idx_train_rel, idx_val_rel in cv.split(
        np.zeros(len(y_base)),
        y_base,
    ):
        idx_train_global = indices_base[idx_train_rel]
        idx_val_global = indices_base[idx_val_rel]

        preds, probs = entrenar_y_predecir_base(
            variante=variante,
            X_por_prep=X_por_prep,
            y=y,
            idx_train=idx_train_global,
            idx_val=idx_val_global,
        )

        for modelo in ORDEN_MODELOS:
            oof_pred[modelo][idx_val_rel] = preds[modelo]
            oof_prob[modelo][idx_val_rel] = probs[modelo]

    y_true = y_base.to_numpy()
    resultados = []

    for candidato in CANDIDATOS_VOTING:
        y_pred = aplicar_voting(
            candidato,
            oof_pred,
            oof_prob,
        )

        resultados.append({
            "nombre": candidato["nombre"],
            "tipo": candidato["tipo"],
            "weights": candidato["weights"],
            "f1_macro": f1_score(
                y_true,
                y_pred,
                average="macro",
            ),
        })

    df = pd.DataFrame(resultados).sort_values(
        ["f1_macro", "nombre"],
        ascending=[False, True],
    ).reset_index(drop=True)

    ganador_nombre = df.loc[0, "nombre"]

    ganador = next(
        c
        for c in CANDIDATOS_VOTING
        if c["nombre"] == ganador_nombre
    )

    return ganador, df


## 7. Nested CV 5×5 del Voting

En cada Outer Fold:

1. El Outer Train entra al Inner CV.
2. En cada Inner Fold se entrenan LR, RF y XGBoost desde cero.
3. Se generan predicciones out-of-fold.
4. Con esas predicciones se escoge hard/soft y los pesos.
5. Se vuelven a entrenar LR, RF y XGBoost con todo el Outer Train.
6. El Voting ganador se evalúa sobre el Outer Test.

El Outer Test nunca participa en la selección del Voting.


In [7]:
def nested_cv_voting_variante(variante):
    X_por_prep, y = cargar_todos_preps(
        variante,
        split="train",
    )

    outer_cv = StratifiedKFold(
        n_splits=N_OUTER,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    resultados_outer = []
    X_ref = X_por_prep["normal"]

    for outer_fold, (idx_outer_train, idx_outer_test) in enumerate(
        outer_cv.split(X_ref, y),
        start=1,
    ):
        print(f"\n{'='*78}")
        print(
            f"{variante.upper()} — VOTING OUTER FOLD "
            f"{outer_fold}/{N_OUTER}"
        )
        print(f"{'='*78}")

        mejor_voting, df_inner = seleccionar_mejor_voting_oof(
            variante=variante,
            X_por_prep=X_por_prep,
            y=y,
            indices_base=idx_outer_train,
            n_splits=N_INNER,
            seed=RANDOM_STATE + outer_fold,
        )

        print("\nCandidatos Inner:")
        display(
            df_inner.style.format({
                "f1_macro": "{:.4f}",
            })
        )
        print("\nGanador Inner:", mejor_voting)

        modelos = crear_modelos_base(variante)
        preds_outer = {}
        probs_outer = {}
        f1_base = {}

        y_outer = y.iloc[idx_outer_test].to_numpy()

        for nombre in ORDEN_MODELOS:
            prep = prep_modelo(variante, nombre)

            X_train = (
                X_por_prep[prep]
                .iloc[idx_outer_train]
                .reset_index(drop=True)
            )
            X_test = (
                X_por_prep[prep]
                .iloc[idx_outer_test]
                .reset_index(drop=True)
            )
            y_train = y.iloc[idx_outer_train].to_numpy()

            modelo = modelos[nombre]
            modelo.fit(X_train, y_train)

            preds_outer[nombre] = modelo.predict(X_test)
            probs_outer[nombre] = probabilidad_clase_1(
                modelo,
                X_test,
            )

            f1_base[nombre] = f1_score(
                y_outer,
                preds_outer[nombre],
                average="macro",
            )

        y_pred_voting = aplicar_voting(
            mejor_voting,
            preds_outer,
            probs_outer,
        )

        f1_voting = f1_score(
            y_outer,
            y_pred_voting,
            average="macro",
        )

        resultados_outer.append({
            "variante": variante,
            "outer_fold": outer_fold,
            "voting": mejor_voting["nombre"],
            "tipo": mejor_voting["tipo"],
            "weights": str(mejor_voting["weights"]),
            "inner_f1_voting": float(df_inner.loc[0, "f1_macro"]),
            "f1_lr": f1_base["LR"],
            "f1_rf": f1_base["RF"],
            "f1_xgb": f1_base["XGB"],
            "f1_voting": f1_voting,
        })

        print(f"\nF1 Outer LR     : {f1_base['LR']:.4f}")
        print(f"F1 Outer RF     : {f1_base['RF']:.4f}")
        print(f"F1 Outer XGB    : {f1_base['XGB']:.4f}")
        print(f"F1 Outer VOTING : {f1_voting:.4f}")

    df_outer = pd.DataFrame(resultados_outer)

    resumen = {
        "variante": variante,
        "f1_voting_mean": df_outer["f1_voting"].mean(),
        "f1_voting_std": df_outer["f1_voting"].std(ddof=1),
        "f1_voting_min": df_outer["f1_voting"].min(),
        "f1_voting_max": df_outer["f1_voting"].max(),
        "f1_lr_mean": df_outer["f1_lr"].mean(),
        "f1_rf_mean": df_outer["f1_rf"].mean(),
        "f1_xgb_mean": df_outer["f1_xgb"].mean(),
    }

    return df_outer, resumen


## 8. Ejecutar Nested CV para MX, ES y CU

In [8]:
VOTING_DETALLE = {}
VOTING_RESUMEN = []

for variante in VARIANTES:
    detalle, resumen = nested_cv_voting_variante(variante)

    VOTING_DETALLE[variante] = detalle
    VOTING_RESUMEN.append(resumen)

    detalle.to_csv(
        f"{RESULTS_DIR}/voting_outer_folds_{variante}.csv",
        index=False,
    )

df_voting_resumen = pd.DataFrame(VOTING_RESUMEN)

print("\nRESUMEN NESTED CV DEL VOTING")
display(
    df_voting_resumen.style.format({
        "f1_voting_mean": "{:.4f}",
        "f1_voting_std": "{:.4f}",
        "f1_voting_min": "{:.4f}",
        "f1_voting_max": "{:.4f}",
        "f1_lr_mean": "{:.4f}",
        "f1_rf_mean": "{:.4f}",
        "f1_xgb_mean": "{:.4f}",
    })
)

df_voting_resumen.to_csv(
    f"{RESULTS_DIR}/voting_nestedcv_resumen.csv",
    index=False,
)



MX — VOTING OUTER FOLD 1/5

Candidatos Inner:


,nombre,tipo,weights,f1_macro
0,hard_equal,hard,"(1, 1, 1)",0.6415
1,soft_lr_rf,soft,"(2, 2, 1)",0.6369
2,soft_lr2,soft,"(2, 1, 1)",0.6366
3,soft_rf2,soft,"(1, 2, 1)",0.6294
4,soft_equal,soft,"(1, 1, 1)",0.6275
5,soft_lr_xgb,soft,"(2, 1, 2)",0.6268
6,soft_rf_xgb,soft,"(1, 2, 2)",0.6140
7,soft_xgb2,soft,"(1, 1, 2)",0.6108



Ganador Inner: {'nombre': 'hard_equal', 'tipo': 'hard', 'weights': (1, 1, 1)}

F1 Outer LR     : 0.6757
F1 Outer RF     : 0.6402
F1 Outer XGB    : 0.6016
F1 Outer VOTING : 0.6596

MX — VOTING OUTER FOLD 2/5

Candidatos Inner:


,nombre,tipo,weights,f1_macro
0,soft_lr_rf,soft,"(2, 2, 1)",0.6593
1,soft_lr2,soft,"(2, 1, 1)",0.6557
2,hard_equal,hard,"(1, 1, 1)",0.6499
3,soft_rf2,soft,"(1, 2, 1)",0.6481
4,soft_equal,soft,"(1, 1, 1)",0.6396
5,soft_lr_xgb,soft,"(2, 1, 2)",0.6364
6,soft_rf_xgb,soft,"(1, 2, 2)",0.6304
7,soft_xgb2,soft,"(1, 1, 2)",0.6285



Ganador Inner: {'nombre': 'soft_lr_rf', 'tipo': 'soft', 'weights': (2, 2, 1)}

F1 Outer LR     : 0.6522
F1 Outer RF     : 0.6457
F1 Outer XGB    : 0.6332
F1 Outer VOTING : 0.6519

MX — VOTING OUTER FOLD 3/5

Candidatos Inner:


...

,nombre,tipo,weights,f1_macro
0,soft_lr2,soft,"(2, 1, 1)",0.6761
1,soft_lr_xgb,soft,"(2, 1, 2)",0.6761
2,soft_lr_rf,soft,"(2, 2, 1)",0.6756
3,soft_equal,soft,"(1, 1, 1)",0.6753
4,soft_rf2,soft,"(1, 2, 1)",0.6740
5,hard_equal,hard,"(1, 1, 1)",0.6717
6,soft_xgb2,soft,"(1, 1, 2)",0.6701
7,soft_rf_xgb,soft,"(1, 2, 2)",0.6694



Ganador Inner: {'nombre': 'soft_lr2', 'tipo': 'soft', 'weights': (2, 1, 1)}

F1 Outer LR     : 0.6683
F1 Outer RF     : 0.6715
F1 Outer XGB    : 0.6586
F1 Outer VOTING : 0.6679

CU — VOTING OUTER FOLD 5/5

Candidatos Inner:


,nombre,tipo,weights,f1_macro
0,hard_equal,hard,"(1, 1, 1)",0.6811
1,soft_lr2,soft,"(2, 1, 1)",0.6794
2,soft_lr_rf,soft,"(2, 2, 1)",0.6789
3,soft_equal,soft,"(1, 1, 1)",0.6765
4,soft_lr_xgb,soft,"(2, 1, 2)",0.6754
5,soft_rf2,soft,"(1, 2, 1)",0.6744
6,soft_xgb2,soft,"(1, 1, 2)",0.6720
7,soft_rf_xgb,soft,"(1, 2, 2)",0.6707



Ganador Inner: {'nombre': 'hard_equal', 'tipo': 'hard', 'weights': (1, 1, 1)}

F1 Outer LR     : 0.6361
F1 Outer RF     : 0.6372
F1 Outer XGB    : 0.6446
F1 Outer VOTING : 0.6464

RESUMEN NESTED CV DEL VOTING


,variante,f1_voting_mean,f1_voting_std,f1_voting_min,f1_voting_max,f1_lr_mean,f1_rf_mean,f1_xgb_mean
0,mx,0.6603,0.0210,0.6320,0.6888,0.6666,0.6480,0.6244
1,es,0.7168,0.0206,0.6978,0.7442,0.7120,0.7072,0.7049
2,cu,0.6704,0.0168,0.6464,0.6924,0.6756,0.6695,0.6650


## 9. Detalle de los 5 Outer Folds y media ± desviación estándar

In [9]:
for variante in VARIANTES:
    df = VOTING_DETALLE[variante]

    print(f"\n{'='*90}")
    print(f"OUTER FOLDS — {variante.upper()}")
    print(f"{'='*90}")

    display(
        df.style.format({
            "inner_f1_voting": "{:.4f}",
            "f1_lr": "{:.4f}",
            "f1_rf": "{:.4f}",
            "f1_xgb": "{:.4f}",
            "f1_voting": "{:.4f}",
        })
    )

    media = df["f1_voting"].mean()
    std = df["f1_voting"].std(ddof=1)

    print(
        f"\n{variante.upper()} — "
        f"Voting F1-Macro = {media:.4f} ± {std:.4f}"
    )



OUTER FOLDS — MX


,variante,outer_fold,voting,tipo,weights,inner_f1_voting,f1_lr,f1_rf,f1_xgb,f1_voting
0,mx,1,hard_equal,hard,"(1, 1, 1)",0.6415,0.6757,0.6402,0.6016,0.6596
1,mx,2,soft_lr_rf,soft,"(2, 2, 1)",0.6593,0.6522,0.6457,0.6332,0.6519
2,mx,3,soft_lr2,soft,"(2, 1, 1)",0.6362,0.6721,0.6690,0.6394,0.6694
3,mx,4,soft_lr_rf,soft,"(2, 2, 1)",0.6568,0.6708,0.6540,0.6223,0.6888
4,mx,5,soft_lr2,soft,"(2, 1, 1)",0.6507,0.6624,0.6310,0.6254,0.6320



MX — Voting F1-Macro = 0.6603 ± 0.0210

OUTER FOLDS — ES


,variante,outer_fold,voting,tipo,weights,inner_f1_voting,f1_lr,f1_rf,f1_xgb,f1_voting
0,es,1,soft_xgb2,soft,"(1, 1, 2)",0.7292,0.7051,0.6959,0.6851,0.7000
1,es,2,soft_lr2,soft,"(2, 1, 1)",0.7250,0.6804,0.6911,0.7236,0.7094
2,es,3,soft_lr_rf,soft,"(2, 2, 1)",0.7159,0.7392,0.7324,0.7129,0.7328
3,es,4,soft_rf2,soft,"(1, 2, 1)",0.7272,0.6886,0.6797,0.6791,0.6978
4,es,5,soft_equal,soft,"(1, 1, 1)",0.7114,0.7466,0.7369,0.7240,0.7442



ES — Voting F1-Macro = 0.7168 ± 0.0206

OUTER FOLDS — CU


,variante,outer_fold,voting,tipo,weights,inner_f1_voting,f1_lr,f1_rf,f1_xgb,f1_voting
0,cu,1,soft_xgb2,soft,"(1, 1, 2)",0.6554,0.7103,0.7088,0.6844,0.6924
1,cu,2,soft_lr2,soft,"(2, 1, 1)",0.6791,0.6667,0.6686,0.6844,0.6776
2,cu,3,soft_rf_xgb,soft,"(1, 2, 2)",0.6838,0.6967,0.6614,0.6532,0.6676
3,cu,4,soft_lr2,soft,"(2, 1, 1)",0.6761,0.6683,0.6715,0.6586,0.6679
4,cu,5,hard_equal,hard,"(1, 1, 1)",0.6811,0.6361,0.6372,0.6446,0.6464



CU — Voting F1-Macro = 0.6704 ± 0.0168


## 10. Selección final del Voting usando todo el TRAIN

Antes de abrir el test se realiza un último 5-fold CV sobre todo el train.

Los modelos base permanecen fijos. Solo se selecciona:

- Hard vs soft.
- Los pesos del soft voting.

El resultado queda congelado antes de utilizar el test oficial.


In [10]:
VOTING_FINAL = {}

for variante in VARIANTES:
    X_por_prep, y = cargar_todos_preps(
        variante,
        split="train",
    )

    indices = np.arange(len(y))

    ganador, df_candidatos = seleccionar_mejor_voting_oof(
        variante=variante,
        X_por_prep=X_por_prep,
        y=y,
        indices_base=indices,
        n_splits=5,
        seed=RANDOM_STATE,
    )

    VOTING_FINAL[variante] = {
        "config": ganador,
        "cv_f1": float(df_candidatos.loc[0, "f1_macro"]),
        "tabla": df_candidatos,
    }

    print(f"\n{'='*78}")
    print(f"CONFIGURACIÓN FINAL VOTING — {variante.upper()}")
    print(f"{'='*78}")

    display(
        df_candidatos.style.format({
            "f1_macro": "{:.4f}",
        })
    )

    print("\nGANADOR:", ganador)

    with open(
        f"{RESULTS_DIR}/voting_config_final_{variante}.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "variante": variante,
                "voting": ganador,
                "f1_cv": VOTING_FINAL[variante]["cv_f1"],
            },
            f,
            indent=4,
            ensure_ascii=False,
        )



CONFIGURACIÓN FINAL VOTING — MX


,nombre,tipo,weights,f1_macro
0,hard_equal,hard,"(1, 1, 1)",0.6617
1,soft_lr2,soft,"(2, 1, 1)",0.6615
2,soft_lr_rf,soft,"(2, 2, 1)",0.6602
3,soft_rf2,soft,"(1, 2, 1)",0.6561
4,soft_equal,soft,"(1, 1, 1)",0.6523
5,soft_lr_xgb,soft,"(2, 1, 2)",0.6499
6,soft_rf_xgb,soft,"(1, 2, 2)",0.6445
7,soft_xgb2,soft,"(1, 1, 2)",0.6382



GANADOR: {'nombre': 'hard_equal', 'tipo': 'hard', 'weights': (1, 1, 1)}

CONFIGURACIÓN FINAL VOTING — ES


,nombre,tipo,weights,f1_macro
0,hard_equal,hard,"(1, 1, 1)",0.7220
1,soft_lr_rf,soft,"(2, 2, 1)",0.7191
2,soft_rf2,soft,"(1, 2, 1)",0.7190
3,soft_lr_xgb,soft,"(2, 1, 2)",0.7182
4,soft_xgb2,soft,"(1, 1, 2)",0.7176
5,soft_rf_xgb,soft,"(1, 2, 2)",0.7175
6,soft_lr2,soft,"(2, 1, 1)",0.7166
7,soft_equal,soft,"(1, 1, 1)",0.7142



GANADOR: {'nombre': 'hard_equal', 'tipo': 'hard', 'weights': (1, 1, 1)}

CONFIGURACIÓN FINAL VOTING — CU


,nombre,tipo,weights,f1_macro
0,hard_equal,hard,"(1, 1, 1)",0.6783
1,soft_rf2,soft,"(1, 2, 1)",0.6783
2,soft_rf_xgb,soft,"(1, 2, 2)",0.6773
3,soft_lr_xgb,soft,"(2, 1, 2)",0.6769
4,soft_xgb2,soft,"(1, 1, 2)",0.6769
5,soft_lr2,soft,"(2, 1, 1)",0.6757
6,soft_equal,soft,"(1, 1, 1)",0.6746
7,soft_lr_rf,soft,"(2, 2, 1)",0.6723



GANADOR: {'nombre': 'hard_equal', 'tipo': 'hard', 'weights': (1, 1, 1)}


## 11. Entrenamiento final y TEST oficial

**A partir de esta sección se abre el test oficial.**

Para cada variante:

1. LR, RF y XGBoost se entrenan con todo el train.
2. Cada uno utiliza su preprocessing correspondiente.
3. Se combinan con el Voting ya seleccionado.
4. Se reporta el resultado del ensemble.
5. Los resultados individuales se muestran solo como comparación final.

No se modifica ninguna configuración después de observar el test.


In [11]:
RESULTADOS_TEST_VOTING = []
MODELOS_VOTING_FINALES = {}

for variante in VARIANTES:
    print(f"\n{'#'*82}")
    print(f"TEST OFICIAL VOTING — {variante.upper()}")
    print(f"{'#'*82}")

    X_train_preps, y_train = cargar_todos_preps(
        variante,
        split="train",
    )
    X_test_preps, y_test = cargar_todos_preps(
        variante,
        split="test",
    )

    modelos = crear_modelos_base(variante)
    pred_test = {}
    prob_test = {}
    f1_individual = {}

    for nombre in ORDEN_MODELOS:
        prep = prep_modelo(variante, nombre)

        X_train = X_train_preps[prep]
        X_test = X_test_preps[prep]

        modelo = modelos[nombre]
        modelo.fit(
            X_train,
            y_train.to_numpy(),
        )

        pred_test[nombre] = modelo.predict(X_test)
        prob_test[nombre] = probabilidad_clase_1(
            modelo,
            X_test,
        )

        f1_individual[nombre] = f1_score(
            y_test,
            pred_test[nombre],
            average="macro",
        )

    config = VOTING_FINAL[variante]["config"]

    y_pred_voting = aplicar_voting(
        config,
        pred_test,
        prob_test,
    )

    resultado = {
        "variante": variante,
        "voting": config["nombre"],
        "tipo": config["tipo"],
        "weights": str(config["weights"]),
        "f1_cv_voting_final": VOTING_FINAL[variante]["cv_f1"],
        "f1_test_lr": f1_individual["LR"],
        "f1_test_rf": f1_individual["RF"],
        "f1_test_xgb": f1_individual["XGB"],
        "f1_macro_test_voting": f1_score(
            y_test,
            y_pred_voting,
            average="macro",
        ),
        "accuracy_test_voting": accuracy_score(
            y_test,
            y_pred_voting,
        ),
        "precision_macro_test_voting": precision_score(
            y_test,
            y_pred_voting,
            average="macro",
            zero_division=0,
        ),
        "recall_macro_test_voting": recall_score(
            y_test,
            y_pred_voting,
            average="macro",
            zero_division=0,
        ),
    }

    RESULTADOS_TEST_VOTING.append(resultado)

    MODELOS_VOTING_FINALES[variante] = {
        "LR": modelos["LR"],
        "RF": modelos["RF"],
        "XGB": modelos["XGB"],
        "voting_config": config,
    }

    print("Configuración:", config)
    print(f"LR Test F1     : {f1_individual['LR']:.4f}")
    print(f"RF Test F1     : {f1_individual['RF']:.4f}")
    print(f"XGB Test F1    : {f1_individual['XGB']:.4f}")
    print(f"VOTING Test F1 : {resultado['f1_macro_test_voting']:.4f}")

    print("\nClassification report — VOTING:")
    print(
        classification_report(
            y_test,
            y_pred_voting,
            target_names=["No irónico", "Irónico"],
            digits=4,
            zero_division=0,
        )
    )

    print("Matriz de confusión — VOTING:")
    print(confusion_matrix(y_test, y_pred_voting))

    joblib.dump(
        MODELOS_VOTING_FINALES[variante],
        f"{MODELS_DIR}/voting_{variante}.pkl",
    )

df_test_voting = pd.DataFrame(RESULTADOS_TEST_VOTING)

print("\nRESULTADOS FINALES VOTING")
display(
    df_test_voting.style.format({
        "f1_cv_voting_final": "{:.4f}",
        "f1_test_lr": "{:.4f}",
        "f1_test_rf": "{:.4f}",
        "f1_test_xgb": "{:.4f}",
        "f1_macro_test_voting": "{:.4f}",
        "accuracy_test_voting": "{:.4f}",
        "precision_macro_test_voting": "{:.4f}",
        "recall_macro_test_voting": "{:.4f}",
    })
)

df_test_voting.to_csv(
    f"{RESULTS_DIR}/voting_test_final.csv",
    index=False,
)



##################################################################################
TEST OFICIAL VOTING — MX
##################################################################################
Configuración: {'nombre': 'hard_equal', 'tipo': 'hard', 'weights': (1, 1, 1)}
LR Test F1     : 0.6924
RF Test F1     : 0.6255
XGB Test F1    : 0.6230
VOTING Test F1 : 0.6507

Classification report — VOTING:
              precision    recall  f1-score   support

  No irónico     0.7702    0.7606    0.7654       401
     Irónico     0.5294    0.5427    0.5360       199

    accuracy                         0.6883       600
   macro avg     0.6498    0.6517    0.6507       600
weighted avg     0.6903    0.6883    0.6893       600

Matriz de confusión — VOTING:
[[305  96]
 [ 91 108]]

##################################################################################
TEST OFICIAL VOTING — ES
##################################################################################
Configuración: {'nombre': 'ha

,variante,voting,tipo,weights,f1_cv_voting_final,f1_test_lr,f1_test_rf,f1_test_xgb,f1_macro_test_voting,accuracy_test_voting,precision_macro_test_voting,recall_macro_test_voting
0,mx,hard_equal,hard,"(1, 1, 1)",0.6617,0.6924,0.6255,0.6230,0.6507,0.6883,0.6498,0.6517
1,es,hard_equal,hard,"(1, 1, 1)",0.7220,0.7179,0.6883,0.6918,0.6932,0.7283,0.6940,0.6925
2,cu,hard_equal,hard,"(1, 1, 1)",0.6783,0.6718,0.6796,0.6551,0.6786,0.7300,0.6946,0.6713


## 12. Comparación final: LR vs RF vs XGBoost vs Voting

In [12]:
filas_comparacion = []

for r in RESULTADOS_TEST_VOTING:
    filas_comparacion.extend([
        {
            "Variante": r["variante"],
            "Modelo": "LR",
            "F1-Macro test": r["f1_test_lr"],
        },
        {
            "Variante": r["variante"],
            "Modelo": "RF",
            "F1-Macro test": r["f1_test_rf"],
        },
        {
            "Variante": r["variante"],
            "Modelo": "XGBoost",
            "F1-Macro test": r["f1_test_xgb"],
        },
        {
            "Variante": r["variante"],
            "Modelo": "Voting",
            "F1-Macro test": r["f1_macro_test_voting"],
        },
    ])

df_comparacion_final = pd.DataFrame(filas_comparacion)

tabla = df_comparacion_final.pivot(
    index="Variante",
    columns="Modelo",
    values="F1-Macro test",
)

display(tabla.style.format("{:.4f}"))

df_comparacion_final.to_csv(
    f"{RESULTS_DIR}/comparacion_lr_rf_xgb_voting.csv",
    index=False,
)


Modelo,LR,RF,Voting,XGBoost
Variante,,,,
cu,0.6718,0.6796,0.6786,0.6551
es,0.7179,0.6883,0.6932,0.6918
mx,0.6924,0.6255,0.6507,0.6230


## 13. Resumen metodológico

```text
Configuraciones base fijadas
LR + RF + XGBoost
        │
        ▼
TRAIN oficial
        │
        ▼
Outer CV 5-fold
        │
        ├── Outer Train
        │       │
        │       ▼
        │   Inner CV 5-fold
        │       │
        │       ├── entrenar LR
        │       ├── entrenar RF
        │       └── entrenar XGB
        │               │
        │               ▼
        │       seleccionar Voting
        │       hard / soft / pesos
        │
        ▼
refit LR + RF + XGB
sobre todo Outer Train
        │
        ▼
Outer Test
        │
        ▼
F1-Macro Voting
        │
        ▼
5 resultados externos
        │
        ▼
media ± desviación estándar
        │
        ▼
selección final Voting
con todo TRAIN
        │
        ▼
TEST OFICIAL
una sola vez
```
